In [1]:
import geopandas as gpd, pandas as pd

panel = pd.read_stata("/shared/share_hle/data/aux_data/global_mortality_panel_public.dta")
# EU recode (Kevin’s set incl. FRA, LIE, TUR)
EU = {"AUT","BEL","BGR","CHE","CYP","CZE","DEU","DNK","ESP",
      "EST","FRA","FIN","GBR","GRC","HRV","HUN","IRL","ISL","ITA",
      "LIE","LTU","LUX","LVA","MKD","MLT","MNE","NLD","NOR",
      "POL","PRT","ROU","SVK","SVN","SWE","TUR"}
panel = panel.assign(iso0=panel["iso"])
panel["iso"] = panel["iso0"].where(~panel["iso0"].isin(EU), "EU")

keys_panel = panel[["iso","adm1_id","adm2_id"]].drop_duplicates()

gdf = gpd.read_file("/shared/share_hle/data/1_estimation/3_regions/insample_shp/mortality_insample_world.shp")
gdf = gdf.to_crs("EPSG:4326")
gdf["geometry"] = gdf["geometry"].make_valid()

keys_shp = gdf[["iso","adm1_id","adm2_id"]].drop_duplicates()

print("Panel keys:", len(keys_panel))
print("Shp   keys:", len(keys_shp))
print("Panel not in shp:", len(pd.merge(keys_panel, keys_shp, on=["iso","adm1_id","adm2_id"], how="left", indicator=True).query("_merge=='left_only'")))
print("Shp not in panel:", len(pd.merge(keys_shp, keys_panel, on=["iso","adm1_id","adm2_id"], how="left", indicator=True).query("_merge=='left_only'")))


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/user/ab5405/.conda/envs/hle_iv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 2194, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/user/ab5405/.conda/envs/hle_iv/lib/python3.12/site-packages/IPython/core/ultratb.py", line 1182, in structured_traceback
  File "/user/ab5405/.conda/envs/hle_iv/lib/python3.12/site-packages/IPython/core/ultratb.py", line 1053, in structured_traceback
  File "/user/ab5405/.conda/envs/hle_iv/lib/python3.12/site-packages/IPython/core/ultratb.py", line 861, in structured_traceback
  File "/user/ab5405/.conda/envs/hle_iv/lib/python3.12/site-packages/IPython/core/ultratb.py", line 746, in format_exception_as_a_whole
  File "/user/ab5405/.conda/envs/hle_iv/lib/python3.12/site-packages/IPython/core/ultratb.py", line 848, in get_records
  File "/user/ab5405/.conda/envs/hle_iv/lib/python3.12/site-packages/stack_data/core.p

In [7]:
import xarray as xr, numpy as np, pandas as pd

path = "/shared/share_hle/data/climate_raw/MERRA2/tas_day_MERRA2_historical_reanalysis_19800101-20240801.nc"
ds   = xr.open_dataset(path)

# 1. look at the first few time stamps
print(ds.time.values[:5])

# 2. compute the unique time deltas
freqs = pd.to_timedelta(np.unique(ds.time.diff("time").values))
print("Unique time interval(s):", freqs)



['1980-01-01T12:00:00.000000000' '1980-01-02T12:00:00.000000000'
 '1980-01-03T12:00:00.000000000' '1980-01-04T12:00:00.000000000'
 '1980-01-05T12:00:00.000000000']
Unique time interval(s): TimedeltaIndex(['0 days 13:30:00', '1 days 00:00:00', '1 days 10:30:00'], dtype='timedelta64[ns]', freq=None)


In [10]:
import pandas as pd

era  = pd.read_csv("/user/ab5405/summeraliaclimate/code/regressions/0_generate_obs_data/obs_csvs/ERA5_025_by_region_year.csv")
m2   = pd.read_csv("/user/ab5405/summeraliaclimate/code/regressions/0_generate_obs_data/obs_csvs/MERRA2_by_region_year.csv")

tmp  = era.merge(m2, on=["iso","adm2_id","year"], suffixes=("_era","_m2"))
tmp["ratio"] = tmp["tavg_poly_1_MERRA2_GMFD"] / tmp["tavg_poly_1_ERA5_025"]
print(tmp["ratio"].describe())


/tmp/8272630.1.debian.q/ipykernel_92269/2634485512.py:3: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  era  = pd.read_csv("/user/ab5405/summeraliaclimate/code/regressions/0_generate_obs_data/obs_csvs/ERA5_025_by_region_year.csv")
/tmp/8272630.1.debian.q/ipykernel_92269/2634485512.py:4: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  m2   = pd.read_csv("/user/ab5405/summeraliaclimate/code/regressions/0_generate_obs_data/obs_csvs/MERRA2_by_region_year.csv")


MemoryError: Unable to allocate 403. MiB for an array with shape (14, 3768600) and data type float64